# Baseline real: TF-IDF + Logistic Regression

Este notebook implementa un baseline real para clasificación de comentarios tóxicos usando TF-IDF y regresión logística. El objetivo es superar el baseline trivial y establecer una referencia sólida para modelos más avanzados.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
# Cargar el dataset limpio (NO el augmentado, para evitar data leakage)
df = pd.read_csv('../data/processed/youtoxic_clean.csv')
print(f'Total de registros: {len(df)}')
print(f'Distribución de clases:')
print(df['IsToxic'].value_counts())
df.head()

Total de registros: 1000
Distribución de clases:
IsToxic
False    538
True     462
Name: count, dtype: int64


,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,if only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,dont you reckon them black lives matter banner...,True,True,False,False,True,False,False,False,False,False,False,False
3,there are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,the arab dude is absolutely right he should ha...,False,False,False,False,False,False,False,False,False,False,False,False


In [3]:
# Separar variables
X = df['Text']
y = df['IsToxic']

# Dividir en train y test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Tamaño train: {len(X_train)}, test: {len(X_test)}')

Tamaño train: 800, test: 200


## Preprocesamiento adicional para TF-IDF

Para modelos clásicos como TF-IDF + Logistic Regression, es recomendable:
- **Eliminar stopwords**: Palabras sin significado semántico (the, is, a...)
- **Lematización**: Reducir palabras a su raíz (running → run)
- **Convertir emojis a texto**: 😀 → "happy_face"

In [4]:
# Descargar recursos de NLTK (ejecutar PRIMERO, solo una vez)
import nltk
print("Descargando recursos de NLTK...")
nltk.download('stopwords', download_dir='/Users/ciprian/nltk_data')
nltk.download('wordnet', download_dir='/Users/ciprian/nltk_data')
nltk.download('omw-1.4', download_dir='/Users/ciprian/nltk_data')
print("✅ Recursos descargados")

Descargando recursos de NLTK...
✅ Recursos descargados
✅ Recursos descargados


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ciprian/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/ciprian/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/ciprian/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
import emoji
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Inicializar (los recursos ya deben estar descargados de la celda anterior)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_for_tfidf(text):
    """
    Preprocesamiento optimizado para TF-IDF:
    1. Convertir emojis a texto
    2. Eliminar stopwords
    3. Lematizar
    """
    # 1. Convertir emojis a texto (😀 → :grinning_face:)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 2. Tokenizar y procesar
    words = text.lower().split()
    
    # 3. Eliminar stopwords y lematizar
    processed_words = []
    for word in words:
        # Limpiar caracteres especiales
        word = re.sub(r'[^a-zA-Z_]', '', word)
        if word and word not in stop_words:
            processed_words.append(lemmatizer.lemmatize(word))
    
    return ' '.join(processed_words)

# Ejemplo
ejemplo = X_train.iloc[0]
print('Original:', ejemplo)
print('Procesado:', preprocess_for_tfidf(ejemplo))

Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and take it and let the cops just turn into one massive street gang who are allowed to kill without consequence even a rat when its cornered will defend itself
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend


In [6]:
# Aplicar preprocesamiento a train y test
print('Aplicando preprocesamiento a train...')
X_train_processed = X_train.apply(preprocess_for_tfidf)

print('Aplicando preprocesamiento a test...')
X_test_processed = X_test.apply(preprocess_for_tfidf)

print(f'\n✅ Preprocesamiento completado')
print(f'Train: {len(X_train_processed)} textos')
print(f'Test: {len(X_test_processed)} textos')

# Mostrar ejemplo
print('\n--- Ejemplo ---')
print('Original:', X_train.iloc[0])
print('Procesado:', X_train_processed.iloc[0])

Aplicando preprocesamiento a train...
Aplicando preprocesamiento a test...

✅ Preprocesamiento completado
Train: 800 textos
Test: 200 textos

--- Ejemplo ---
Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and take it and let the cops just turn into one massive street gang who are allowed to kill without consequence even a rat when its cornered will defend itself
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend
Aplicando preprocesamiento a test...

✅ Preprocesamiento completado
Train: 800 textos
Test: 200 textos

--- Ejemplo ---
Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and tak

## ⚠️ Orden Correcto del Pipeline

**Es crucial seguir este orden para evitar Data Leakage:**

1. ✅ Cargar datos limpios
2. ✅ **SPLIT Train/Test** (antes de augmentation)
3. ✅ Data Augmentation **SOLO en Train**
4. ✅ Vectorización (fit en Train, transform en ambos)
5. ✅ Entrenamiento y evaluación

> Si aumentamos los datos ANTES de dividir, podríamos tener el texto original en Test y su variación en Train. El modelo "haría trampa" memorizando respuestas.

## Data Augmentation (Solo en Train)

Aplicamos técnicas de augmentation SOLO al conjunto de entrenamiento:
- **Back-translation**: Traduce a otro idioma y vuelve (paráfrasis naturales)
- **Random Swap**: Intercambia posiciones de palabras

In [7]:
import random
from deep_translator import GoogleTranslator
from nltk.corpus import wordnet

# --- Funciones de Augmentation ---

def back_translation(text, intermediate_lang='es'):
    """Traduce a otro idioma y vuelve para generar paráfrasis."""
    try:
        translated = GoogleTranslator(source='en', target=intermediate_lang).translate(text)
        back_translated = GoogleTranslator(source=intermediate_lang, target='en').translate(translated)
        return back_translated
    except Exception as e:
        return text  # Si falla, devuelve el original

def random_swap(text, n=2):
    """Intercambia n pares de palabras aleatorias."""
    words = text.split()
    if len(words) < 2:
        return text
    new_words = words.copy()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(new_words)), 2)
        new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return ' '.join(new_words)

def synonym_replacement(text, n=2):
    """Reemplaza n palabras aleatorias por sinónimos usando WordNet."""
    words = text.split()
    if len(words) < 2:
        return text
    
    new_words = words.copy()
    # Seleccionar palabras al azar para reemplazar
    word_indices = list(range(len(words)))
    random.shuffle(word_indices)
    
    replacements = 0
    for idx in word_indices:
        if replacements >= n:
            break
        word = words[idx]
        # Obtener sinónimos de WordNet
        synonyms = []
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                if lemma.name() != word and '_' not in lemma.name():
                    synonyms.append(lemma.name())
        
        if synonyms:
            new_words[idx] = random.choice(synonyms)
            replacements += 1
    
    return ' '.join(new_words)

def augment_text(text):
    """Aplica una técnica aleatoria de augmentation."""
    func = random.choice([back_translation, random_swap, synonym_replacement])
    return func(text)

print("Funciones de augmentation definidas ✓")
print("  - back_translation (español)")
print("  - random_swap")
print("  - synonym_replacement (WordNet) ✨ NUEVO")

Funciones de augmentation definidas ✓
  - back_translation (español)
  - random_swap
  - synonym_replacement (WordNet) ✨ NUEVO


In [16]:
# Aplicar augmentation 2X a TODOS los datos del TRAIN (ya preprocesados)
# Más augmentation = más variabilidad = mejor generalización

train_df = pd.DataFrame({'Text': X_train_processed, 'IsToxic': y_train})

print(f'Train original: {len(train_df)}')
print(f'Distribución original: {train_df["IsToxic"].value_counts().to_dict()}')

# Generar 2 versiones aumentadas por cada texto
augmented_texts = []
augmented_labels = []

print('\nAplicando augmentation 2X a todo el train...')
for i, (text, label) in enumerate(zip(train_df['Text'], train_df['IsToxic'])):
    # Generar 2 versiones aumentadas por texto
    augmented_texts.append(augment_text(text))
    augmented_labels.append(label)
    augmented_texts.append(augment_text(text))  # Segunda versión
    augmented_labels.append(label)
    if (i + 1) % 200 == 0:
        print(f'  Procesados {i + 1}/{len(train_df)}...')

# Combinar train original + augmentados (ahora 3x el tamaño original)
df_augmented = pd.DataFrame({'Text': augmented_texts, 'IsToxic': augmented_labels})
train_final = pd.concat([train_df, df_augmented], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Actualizar X_train e y_train
X_train_aug = train_final['Text']
y_train_aug = train_final['IsToxic']

print(f'\nTrain después de augmentation 2X: {len(train_final)} (3x original)')
print(f'Distribución final: {train_final["IsToxic"].value_counts().to_dict()}')

Train original: 800
Distribución original: {False: 430, True: 370}

Aplicando augmentation 2X a todo el train...
  Procesados 200/800...
  Procesados 200/800...
  Procesados 400/800...
  Procesados 400/800...
  Procesados 600/800...
  Procesados 600/800...
  Procesados 800/800...

Train después de augmentation 2X: 2400 (3x original)
Distribución final: {False: 1290, True: 1110}
  Procesados 800/800...

Train después de augmentation 2X: 2400 (3x original)
Distribución final: {False: 1290, True: 1110}


In [24]:
# Crear y ajustar el vectorizador TF-IDF (fit SOLO en train augmentado)
# Configuración AÚN MÁS SIMPLE para reducir overfitting
vectorizer = TfidfVectorizer(
    max_features=500,   # Solo 500 palabras (menos features = menos overfitting)
    min_df=3,           # Ignorar palabras que aparecen en menos de 3 documentos
    max_df=0.90,        # Ignorar palabras que aparecen en más del 90% de documentos
    ngram_range=(1, 1)  # SOLO unigramas (sin bigramas)
)
X_train_tfidf = vectorizer.fit_transform(X_train_aug)  # Fit en train augmentado
X_test_tfidf = vectorizer.transform(X_test_processed)  # Transform en test preprocesado
print(f'Shape train: {X_train_tfidf.shape}, test: {X_test_tfidf.shape}')
print(f'Vocabulario reducido a {len(vectorizer.vocabulary_)} palabras')

Shape train: (2400, 500), test: (200, 500)
Vocabulario reducido a 500 palabras


In [ ]:
# Entrenar con regularización óptima (mejor balance overfitting/performance)
clf = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    C=0.005,             # Regularización óptima (gap ~9.9%)
    class_weight='balanced'
)

clf.fit(X_train_tfidf, y_train_aug)
print(f'Modelo entrenado con {len(y_train_aug)} muestras')
print(f'Regularización C={clf.C} (óptima), class_weight={clf.class_weight}')

Modelo entrenado con 2400 muestras
Regularización C=0.001 (máxima), class_weight=balanced


In [26]:
# Predecir en train augmentado
y_train_pred = clf.predict(X_train_tfidf)

# Calcular métricas en train
print('--- Métricas en entrenamiento (augmentado) ---')
print('Accuracy:', accuracy_score(y_train_aug, y_train_pred))
print('Precision:', precision_score(y_train_aug, y_train_pred, zero_division=0))
print('Recall:', recall_score(y_train_aug, y_train_pred, zero_division=0))
print('F1-score:', f1_score(y_train_aug, y_train_pred, zero_division=0))
print('\nReporte de clasificación:')
print(classification_report(y_train_aug, y_train_pred, zero_division=0))

--- Métricas en entrenamiento (augmentado) ---
Accuracy: 0.8016666666666666
Precision: 0.8019047619047619
Recall: 0.7585585585585586
F1-score: 0.7796296296296297

Reporte de clasificación:
              precision    recall  f1-score   support

       False       0.80      0.84      0.82      1290
        True       0.80      0.76      0.78      1110

    accuracy                           0.80      2400
   macro avg       0.80      0.80      0.80      2400
weighted avg       0.80      0.80      0.80      2400



In [27]:
# Predecir en test
y_pred = clf.predict(X_test_tfidf)

# Calcular métricas
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, zero_division=0))
print('Recall:', recall_score(y_test, y_pred, zero_division=0))
print('F1-score:', f1_score(y_test, y_pred, zero_division=0))

print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.705
Precision: 0.6853932584269663
Recall: 0.6630434782608695
F1-score: 0.6740331491712708

Reporte de clasificación:
              precision    recall  f1-score   support

       False       0.72      0.74      0.73       108
        True       0.69      0.66      0.67        92

    accuracy                           0.70       200
   macro avg       0.70      0.70      0.70       200
weighted avg       0.70      0.70      0.70       200



In [28]:
# Cargar predicciones del baseline trivial y calcular métricas
df_baseline = pd.read_csv('../data/processed/baseline_trivial_pred.csv')
y_true_baseline = df_baseline['IsToxic']
y_pred_baseline = df_baseline['baseline_pred']

print('--- Baseline trivial ---')
print('Accuracy:', accuracy_score(y_true_baseline, y_pred_baseline))
print('Precision:', precision_score(y_true_baseline, y_pred_baseline, zero_division=0))
print('Recall:', recall_score(y_true_baseline, y_pred_baseline, zero_division=0))
print('F1-score:', f1_score(y_true_baseline, y_pred_baseline, zero_division=0))

print('\nReporte de clasificación:')
print(classification_report(y_true_baseline, y_pred_baseline, zero_division=0))

--- Baseline trivial ---
Accuracy: 0.538
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Reporte de clasificación:
              precision    recall  f1-score   support

       False       0.54      1.00      0.70       538
        True       0.00      0.00      0.00       462

    accuracy                           0.54      1000
   macro avg       0.27      0.50      0.35      1000
weighted avg       0.29      0.54      0.38      1000



In [29]:
# Comparar métricas de baseline trivial y modelo TF-IDF + Logistic Regression
def get_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }

# Métricas baseline trivial
metrics_baseline = get_metrics(y_true_baseline, y_pred_baseline)
# Métricas modelo real (usa y_test, y_pred de la celda anterior)
metrics_model = get_metrics(y_test, y_pred)

import pandas as pd
report = pd.DataFrame([metrics_baseline, metrics_model],
                    index=['Baseline trivial', 'TF-IDF + LogReg'])
print('Comparativa de métricas:')
display(report)

Comparativa de métricas:


,accuracy,precision,recall,f1
Baseline trivial,0.538,0.000000,0.000000,0.000000
TF-IDF + LogReg,0.705,0.685393,0.663043,0.674033


In [30]:
# Comparar métricas de entrenamiento y test (incluyendo diferencias)
metrics_train = get_metrics(y_train_aug, y_train_pred)
metrics_test = get_metrics(y_test, y_pred)

df_compare = pd.DataFrame([metrics_train, metrics_test], index=['Train (aug)', 'Test (original)'])
diff_metrics = df_compare.loc['Train (aug)'] - df_compare.loc['Test (original)']
df_compare.loc['Gap (Train-Test)'] = diff_metrics

print('📊 Comparativa de métricas (Train vs Test):')
display(df_compare)

# Evaluar overfitting
print('\n🎯 Análisis de Overfitting:')
if diff_metrics['f1'] < 0.05:
    print(f"✅ Gap F1 = {diff_metrics['f1']:.3f} (<5%) - No hay overfitting significativo")
else:
    print(f"⚠️ Gap F1 = {diff_metrics['f1']:.3f} (>5%) - Posible overfitting")

📊 Comparativa de métricas (Train vs Test):


,accuracy,precision,recall,f1
Train (aug),0.801667,0.801905,0.758559,0.779630
Test (original),0.705000,0.685393,0.663043,0.674033
Gap (Train-Test),0.096667,0.116512,0.095515,0.105596



🎯 Análisis de Overfitting:
⚠️ Gap F1 = 0.106 (>5%) - Posible overfitting


## Análisis de la comparación entre métricas de entrenamiento y test

A continuación se muestra un breve análisis sobre las diferencias entre las métricas de entrenamiento y test. Si las métricas de entrenamiento son mucho mayores que las de test, puede indicar overfitting. Si ambas son bajas, puede indicar underfitting o un modelo poco expresivo.


## Conclusiones de la comparación

- El baseline trivial predice siempre la clase mayoritaria y sirve como referencia mínima.
- El modelo TF-IDF + Logistic Regression utiliza el texto y, si las métricas son mejores que el baseline, demuestra que el modelo aprende patrones útiles.
- Si tu modelo supera al baseline en F1-score, accuracy, precision y recall, puedes avanzar a modelos más complejos o probar mejoras.
- Si no lo supera, revisa el preprocesamiento, los datos o prueba otros enfoques.

> **Recuerda:** El objetivo es que cualquier modelo real supere claramente al baseline trivial para justificar su uso.